In [1]:
import os
import sys
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd

import seaborn as sns
from scipy import stats
from itertools import chain
import shapely.vectorized as sv

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from playsound import playsound
from scipy.spatial import cKDTree

from scipy.stats import linregress
from sklearn.linear_model import LinearRegression

sys.path.append(os.path.abspath(".."))
from function import DOWN_raw
from function import ART_statistic as ART_sta
from function import ART_downscale as ART_down

import warnings
warnings.filterwarnings('ignore')

playsound is relying on another python subprocess. Please use `pip install pygobject` if you want playsound to run more efficiently.


## Create ENSEMBLE from the median of the others individual ENSEMBLE products

In [2]:
def get_nearest_values(ref_lat2d, ref_lon2d, target_lat2d, target_lon2d, target_data):
    """
    Para cada punto en la malla de referencia, busca el valor más cercano en la malla objetivo.
    """
    ny, nx = ref_lat2d.shape
    ref_points = np.column_stack((ref_lat2d.ravel(), ref_lon2d.ravel()))
    target_points = np.column_stack((target_lat2d.ravel(), target_lon2d.ravel()))
    tree = cKDTree(target_points)
    _, idx = tree.query(ref_points)
    matched_values = target_data.ravel()[idx]
    return matched_values.reshape(ny, nx)

In [3]:
seed = 7
Tr = 50

In [4]:
product, time_reso = 'CHIRPS', '1dy'
dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
DATA_REF = xr.open_dataset(dir_input)
DATA_REF

<xarray.Dataset> Size: 2MB
Dimensions:    (lat: 240, lon: 260)
Coordinates:
  * lat        (lat) float32 960B 36.02 36.07 36.12 36.17 ... 47.87 47.92 47.97
  * lon        (lon) float32 1kB 6.025 6.075 6.125 6.175 ... 18.88 18.93 18.97
Data variables:
    MEVd_Down  (lat, lon) float64 499kB ...
    MEVd_LLc   (lat, lon) float64 499kB ...
    MEVd_LSc   (lat, lon) float64 499kB ...
    MEVd_LTO   (lat, lon) float64 499kB ...
    MEVd_QQc   (lat, lon) float64 499kB ...
Attributes:
    description:  CHIRPS Extreme quantiles corrected for 50 years, applying Q...

In [5]:

product, time_reso = 'CHIRPS', '1dy'
dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
DATA_REF = xr.open_dataset(dir_input)
lons_REF, lats_REF = DATA_REF.lon.values, DATA_REF.lat.values
lon2d_REF, lat2d_REF = np.meshgrid(lons_REF, lats_REF)

product, time_reso = 'CMORPH', '3h'
dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
DATA_SAT1 = xr.open_dataset(dir_input)
lon2d_SAT1, lat2d_SAT1 = np.meshgrid(DATA_SAT1.lon.values, DATA_SAT1.lat.values)

product, time_reso = 'ERA5', '3h'
dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
DATA_SAT2 = xr.open_dataset(dir_input)
lon2d_SAT2, lat2d_SAT2 = np.meshgrid(DATA_SAT2.lon.values, DATA_SAT2.lat.values)

product, time_reso = 'MSWEP', '3h'
dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
DATA_SAT4 = xr.open_dataset(dir_input)
lon2d_SAT4, lat2d_SAT4 = np.meshgrid(DATA_SAT4.lon.values, DATA_SAT4.lat.values)

product, time_reso = 'GSMaP', '3h'
dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
DATA_SAT3 = xr.open_dataset(dir_input)
lon2d_SAT3, lat2d_SAT3 = np.meshgrid(DATA_SAT3.lon.values, DATA_SAT3.lat.values)

product, time_reso = 'IMERG', '1dy'
dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
DATA_SAT5 = xr.open_dataset(dir_input)
lon2d_SAT5, lat2d_SAT5 = np.meshgrid(DATA_SAT5.lon.values, DATA_SAT5.lat.values)

DATA_SAT = [DATA_SAT1, DATA_SAT2, DATA_SAT3, DATA_SAT4, DATA_SAT5]
LATLON_SAT = [
            (lat2d_SAT1, lon2d_SAT1),
            (lat2d_SAT2, lon2d_SAT2),
            (lat2d_SAT3, lon2d_SAT3),
            (lat2d_SAT4, lon2d_SAT4),
            (lat2d_SAT5, lon2d_SAT5),]

ntimes = DATA_REF['MEVd_LTO'].shape[0] 

REFE_remap = np.full((lat2d_REF.shape), np.nan)
SAT_remap = [np.full_like(REFE_remap, np.nan) for _ in range(len(DATA_REF))]



In [7]:
# seeds_list = [7, 19, 31, 53, 89, 127, 211, 307, 401, 509, 613, 727, 839, 947, 1051]
seeds_list = [7, 19, 31, 53, 89, 127, 211, 307]

for seed in seeds_list:
    print(f'Seed: {seed}')

    product, time_reso = 'CHIRPS', '1dy'
    dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
    dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
    DATA_REF = xr.open_dataset(dir_input)
    lons_REF, lats_REF = DATA_REF.lon.values, DATA_REF.lat.values
    lon2d_REF, lat2d_REF = np.meshgrid(lons_REF, lats_REF)

    product, time_reso = 'CMORPH', '3h'
    dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
    dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
    DATA_SAT1 = xr.open_dataset(dir_input)
    lon2d_SAT1, lat2d_SAT1 = np.meshgrid(DATA_SAT1.lon.values, DATA_SAT1.lat.values)

    product, time_reso = 'ERA5', '3h'
    dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
    dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
    DATA_SAT2 = xr.open_dataset(dir_input)
    lon2d_SAT2, lat2d_SAT2 = np.meshgrid(DATA_SAT2.lon.values, DATA_SAT2.lat.values)

    product, time_reso = 'MSWEP', '3h'
    dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
    dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
    DATA_SAT4 = xr.open_dataset(dir_input)
    lon2d_SAT4, lat2d_SAT4 = np.meshgrid(DATA_SAT4.lon.values, DATA_SAT4.lat.values)

    product, time_reso = 'GSMaP', '3h'
    dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
    dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
    DATA_SAT3 = xr.open_dataset(dir_input)
    lon2d_SAT3, lat2d_SAT3 = np.meshgrid(DATA_SAT3.lon.values, DATA_SAT3.lat.values)

    product, time_reso = 'IMERG', '1dy'
    dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
    dir_input = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
    DATA_SAT5 = xr.open_dataset(dir_input)
    lon2d_SAT5, lat2d_SAT5 = np.meshgrid(DATA_SAT5.lon.values, DATA_SAT5.lat.values)

    DATA_SAT = [DATA_SAT1, DATA_SAT2, DATA_SAT3, DATA_SAT4, DATA_SAT5]
    LATLON_SAT = [
                (lat2d_SAT1, lon2d_SAT1),
                (lat2d_SAT2, lon2d_SAT2),
                (lat2d_SAT3, lon2d_SAT3),
                (lat2d_SAT4, lon2d_SAT4),
                (lat2d_SAT5, lon2d_SAT5),]

    ntimes = DATA_REF['MEVd_LTO'].shape[0] 

    REFE_remap = np.full((lat2d_REF.shape), np.nan)
    SAT_remap = [np.full_like(REFE_remap, np.nan) for _ in range(len(DATA_REF))]

    REFE = DATA_REF['MEVd_LTO'].values

    # ==================================================================================================

    def median_sat(PARAM):
        for k, (DATA_k, (lat2d_k, lon2d_k)) in enumerate(zip(DATA_SAT, LATLON_SAT)):
            SAT_k = DATA_k[PARAM].values

            SAT_near = get_nearest_values(lat2d_REF, lon2d_REF, lat2d_k, lon2d_k, SAT_k)

            SAT_remap[k] = SAT_near

        assert all(arr.shape == REFE_remap.shape for arr in SAT_remap)
        stacked_all = np.stack([REFE_remap] + SAT_remap,axis=0)

        Ensemble_mean   = np.nanmean(stacked_all, axis=0)
        Ensemble_median = np.nanmedian(stacked_all, axis=0)

        return Ensemble_mean, Ensemble_median

    MEVd_mean, MEVd_median = median_sat('MEVd_Down')
    MEVd_LLc_mean, MEVd_LLc_median = median_sat('MEVd_LLc')
    MEVd_LSc_mean, MEVd_LSc_median = median_sat('MEVd_LSc')
    MEVd_LTO_mean, MEVd_LTO_median = median_sat('MEVd_LTO')
    MEVd_QQc_mean, MEVd_QQc_median = median_sat('MEVd_QQc')

    # ==================================================================================================
    DOWN_ensemble = xr.Dataset(
        data_vars={
            "MEVd_Down": (("lat","lon"), MEVd_mean), 
            "MEVd_LLc": (("lat","lon"), MEVd_LLc_mean), 
            "MEVd_LSc": (("lat","lon"), MEVd_LSc_mean), 
            "MEVd_LTO": (("lat","lon"), MEVd_LTO_mean), 
            "MEVd_QQc": (("lat","lon"), MEVd_QQc_mean), 
            },
        coords={
            'lat': lats_REF, 
            'lon': lons_REF
            },
            attrs=dict(description=f"Ensemble of Extreme quantiles corrected for {Tr} years, using the quantiles from the other products",))

    DOWN_ensemble.MEVd_Down.attrs["units"] = "mm/day"
    DOWN_ensemble.MEVd_Down.attrs["long_name"] = "Mean Extreme quantiles from other quantiles products"
    DOWN_ensemble.MEVd_Down.attrs["origname"] = "Mean Extreme quantiles"

    DOWN_ensemble.MEVd_LLc.attrs["units"] = "mm/day"
    DOWN_ensemble.MEVd_LLc.attrs["long_name"] = "Mean Extreme quantiles corrected with LLc method from other quantiles products"
    DOWN_ensemble.MEVd_LLc.attrs["origname"] = "Linear Regresion log"

    DOWN_ensemble.MEVd_LSc.attrs["units"] = "mm/day"
    DOWN_ensemble.MEVd_LSc.attrs["long_name"] = "Mean Extreme quantiles corrected with LLc method without using intercept from other quantiles products"
    DOWN_ensemble.MEVd_LSc.attrs["origname"] = "Linear Regresion log without intercept"

    DOWN_ensemble.MEVd_LTO.attrs["units"] = "mm/day"
    DOWN_ensemble.MEVd_LTO.attrs["long_name"] = "Mean Extreme quantiles corrected with LTO method from other quantiles products"
    DOWN_ensemble.MEVd_LTO.attrs["origname"] = "Linear Regresion log without intercept"

    DOWN_ensemble.MEVd_QQc.attrs["units"] = "mm/day"
    DOWN_ensemble.MEVd_QQc.attrs["long_name"] = "Mean Extreme quantiles corrected with QQc method from other quantiles products"
    DOWN_ensemble.MEVd_QQc.attrs["origname"] = "Quantile-Quantile map"

    dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
    PRE_out = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_ENSEMBLEd_mean_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
    print(f'Exportin as: {PRE_out}')
    # DOWN_ensemble.to_netcdf(PRE_out)
    
    # ==================================================================================================
    DOWN_ensemble = xr.Dataset(
        data_vars={
            "MEVd_Down": (("lat","lon"), MEVd_median), 
            "MEVd_LLc": (("lat","lon"), MEVd_LLc_median), 
            "MEVd_LSc": (("lat","lon"), MEVd_LSc_median), 
            "MEVd_LTO": (("lat","lon"), MEVd_LTO_median), 
            "MEVd_QQc": (("lat","lon"), MEVd_QQc_median), 
            },
        coords={
            'lat': lats_REF, 
            'lon': lons_REF
            },
            attrs=dict(description=f"Ensemble of Extreme quantiles corrected for {Tr} years, using the quantiles from the other products",))

    DOWN_ensemble.MEVd_Down.attrs["units"] = "mm/day"
    DOWN_ensemble.MEVd_Down.attrs["long_name"] = "Median Extreme quantiles from other quantiles products"
    DOWN_ensemble.MEVd_Down.attrs["origname"] = "Median Extreme quantiles"

    DOWN_ensemble.MEVd_LLc.attrs["units"] = "mm/day"
    DOWN_ensemble.MEVd_LLc.attrs["long_name"] = "Median Extreme quantiles corrected with LLc method from other quantiles RSR products"
    DOWN_ensemble.MEVd_LLc.attrs["origname"] = "Median Linear Regresion log"

    DOWN_ensemble.MEVd_LSc.attrs["units"] = "mm/day"
    DOWN_ensemble.MEVd_LSc.attrs["long_name"] = "Median Extreme quantiles corrected with LLc method without using intercept from other quantiles RSR products"
    DOWN_ensemble.MEVd_LSc.attrs["origname"] = "Median Linear Regresion log without intercept"

    DOWN_ensemble.MEVd_LTO.attrs["units"] = "mm/day"
    DOWN_ensemble.MEVd_LTO.attrs["long_name"] = "Median Extreme quantiles corrected with LTO method from other quantiles RSR products"
    DOWN_ensemble.MEVd_LTO.attrs["origname"] = "Median Linear Regresion log without intercept"

    DOWN_ensemble.MEVd_QQc.attrs["units"] = "mm/day"
    DOWN_ensemble.MEVd_QQc.attrs["long_name"] = "Median Extreme quantiles corrected with QQc method from other quantiles RSR products"
    DOWN_ensemble.MEVd_QQc.attrs["origname"] = "Median Quantile-Quantile map"

    dir_base = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','QUANTILE')
    PRE_out = os.path.join(os.path.join(dir_base, f'ITALY_DOWN_ENSEMBLEd_median_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_{str(seed).zfill(4)}.nc'))
    print(f'Exportin as: {PRE_out}')
    DOWN_ensemble.to_netcdf(PRE_out)

Seed: 7
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/QUANTILE/ITALY_DOWN_ENSEMBLEd_mean_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_0007.nc
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/QUANTILE/ITALY_DOWN_ENSEMBLEd_median_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_0007.nc
Seed: 19
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/QUANTILE/ITALY_DOWN_ENSEMBLEd_mean_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_0019.nc
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/QUANTILE/ITALY_DOWN_ENSEMBLEd_median_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_0019.nc
Seed: 31
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/QUANTILE/ITALY_DOWN_ENSEMBLEd_mean_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_QQc_LLc_0031.nc
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/QUANTILE/ITALY_DOWN_ENSEMBLEd_median_1dy_2002_